# Оценка качества прогноза количества операций (operations) и нагрузки

Этот ноутбук загружает исторические данные по операциям из
`../data/processed/operations_from_history.csv` и считает метрики (metrics)
для факта (fact) и прогноза ИИ (AI forecast), а также baseline (baseline model).

Цель — получить числовые показатели качества и подготовить данные для
дашборда (dashboard) в `streamlit_app.py`. 


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/processed/operations_from_history.csv")

df_ops = pd.read_csv(DATA_PATH, parse_dates=["period"])

# Переименуем для наглядности: факт и прогноз ИИ (AI forecast)
df_ops = df_ops.rename(columns={
    "operation_fact_value": "fact",
    "operation_forecast_value": "ai_pred",
})

# Месяц (month) для агрегирования
df_ops["month"] = df_ops["period"].dt.to_period("M").dt.to_timestamp()

# Исключаем прогнозные записи (forecast records), если они есть
PREDICT_COMMENT = "first_predict_value_record"
if "history_comment" in df_ops.columns:
    df_ops = df_ops[df_ops["history_comment"] != PREDICT_COMMENT]

df_ops.head()


In [ ]:
def compute_metrics(y_true, y_pred, large_abs=100, large_pct=0.3):
    """Расчёт базовых метрик (metrics) ошибки прогноза.

    y_true — факт (fact), y_pred — прогноз (prediction).
    large_abs — порог по абсолютной ошибке, large_pct — по относительной.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE_%": np.nan, "large_error_share": np.nan}

    error = y_pred - y_true
    abs_error = np.abs(error)

    mae = abs_error.mean()
    rmse = np.sqrt((error ** 2).mean())

    mape = np.nan
    nonzero = y_true != 0
    if nonzero.any():
        mape = (abs_error[nonzero] / np.abs(y_true[nonzero]) * 100).mean()

    large_mask = (abs_error > large_abs) | (abs_error > np.abs(y_true) * large_pct)
    large_share = large_mask.mean()

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_%": mape,
        "large_error_share": large_share,
    }

# Пороги для Operations (operations)
METRICS_KW = dict(large_abs=100, large_pct=0.3)

compute_metrics(df_ops["fact"], df_ops["ai_pred"], **METRICS_KW)


In [ ]:
# Построение baseline (baseline model): факт год назад по той же операции

key = ["resist_id", "operation_id"]

df_base = df_ops.copy()
df_base["month_prev"] = df_base["month"] - pd.DateOffset(months=12)  # сдвиг на год назад

prev = (
    df_base[key + ["month", "fact"]]
    .rename(columns={"month": "month_prev", "fact": "fact_prev"})
)

df_eval = df_base.merge(prev, on=key + ["month_prev"], how="left")
df_eval["baseline_pred"] = df_eval["fact_prev"]

df_eval[["fact", "ai_pred", "baseline_pred"]].head()


In [ ]:
# Глобальные метрики (metrics) по источникам: ai vs baseline

sources = {
    "ai": df_eval["ai_pred"],
    "baseline": df_eval["baseline_pred"],
}

rows = []
for name, pred in sources.items():
    m = compute_metrics(df_eval["fact"], pred, **METRICS_KW)
    m["source"] = name
    rows.append(m)

df_sources = pd.DataFrame(rows).set_index("source")
df_sources


In [ ]:
# Метрики по подразделениям (divisions)

rows = []
for div, grp in df_eval.groupby("division_title"):
    m_ai = compute_metrics(grp["fact"], grp["ai_pred"], **METRICS_KW)
    m_ai.update({"division_title": div, "source": "ai"})
    rows.append(m_ai)

    m_bl = compute_metrics(grp["fact"], grp["baseline_pred"], **METRICS_KW)
    m_bl.update({"division_title": div, "source": "baseline"})
    rows.append(m_bl)

df_div = pd.DataFrame(rows)

if df_div.empty:
    print("Нет данных для расчёта метрик по подразделениям.")
else:
    pivot_div = df_div.pivot_table(
        index="division_title",
        columns="source",
        values=["MAE", "MAPE_%"],
    )

    pivot_div


In [ ]:
# Метрики по месяцам (months) — динамика качества

rows = []
for month, grp in df_eval.groupby("month"):
    for name, col in [("ai", "ai_pred"), ("baseline", "baseline_pred")]:
        m = compute_metrics(grp["fact"], grp[col], **METRICS_KW)
        m.update({"month": month, "source": name})
        rows.append(m)

df_month = pd.DataFrame(rows)

if df_month.empty or "month" not in df_month.columns:
    print("Нет данных для расчёта метрик по месяцам.")
else:
    df_month_pivot = df_month.pivot(index="month", columns="source", values="MAE")
    df_month_pivot.tail()


In [ ]:
# Вспомогательные функции (helper functions) для тепловых карт (heatmaps)

def build_heatmap_pivot(
    df: pd.DataFrame,
    metric: str = "MAE",
    source: str = "ai",
    by: str = "division",
):
    """Строит сводную таблицу (pivot_table) для тепловой карты.

    metric  — одна из метрик: "MAE", "RMSE", "MAPE_%", "large_error_share".
    source  — "ai" или "baseline" (какой прогноз используем).
    by      — "division" (подразделение × месяц) или "resist" (resist_id × месяц).
    """
    assert metric in {"MAE", "RMSE", "MAPE_%", "large_error_share"}
    assert source in {"ai", "baseline"}
    assert by in {"division", "resist"}

    pred_col = "ai_pred" if source == "ai" else "baseline_pred"

    if by == "division":
        group_keys = ["division_title", "month"]
        row_key = "division_title"
    else:
        group_keys = ["resist_id", "month"]
        row_key = "resist_id"

    rows = []
    for keys, grp in df.groupby(group_keys):
        m = compute_metrics(grp["fact"], grp[pred_col], **METRICS_KW)
        row = {
            row_key: keys[0],
            "month": keys[1],
            metric: m[metric],
        }
        rows.append(row)

    df_h = pd.DataFrame(rows)
    if df_h.empty:
        return pd.DataFrame()

    pivot = df_h.pivot(index=row_key, columns="month", values=metric)
    # Для удобства отображения (для heatmap в Plotly/Matplotlib)
    pivot.columns = pivot.columns.astype(str)
    return pivot


# Примеры построения сводных таблиц для тепловых карт (heatmaps)
heatmap_div_ai_mae = build_heatmap_pivot(df_eval, metric="MAE", source="ai", by="division")
heatmap_res_bl_mape = build_heatmap_pivot(df_eval, metric="MAPE_%", source="baseline", by="resist")

heatmap_div_ai_mae.head()
